In [2]:
!pip install transformers sentence-transformers -q

In [18]:
from huggingface_hub import login
login()  # paste your HF token when prompted

In [15]:
import sys
import numpy as np
sys.path.insert(0, r'e:\TalentLens')

from transformers import pipeline
from sentence_transformers import SentenceTransformer
from src.data.preprocess import clean_text

In [20]:
skill_extractor = pipeline(
    task="token-classification",
    model="algiraldohe/lm-ner-linkedin-skills-recognition",
    aggregation_strategy="simple"
)

config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

d:\anaconda2\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\My home PC\.cache\huggingface\hub\models--algiraldohe--lm-ner-linkedin-skills-recognition. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [21]:
sample_text = """We are looking for a Python developer with experience in 
machine learning, Docker, and SQL databases. 
Knowledge of Kubernetes is a plus."""

results = skill_extractor(sample_text)

print("Extracted skills:")
for r in results:
    print(f"  skill: {r['word']:<25} confidence: {r['score']:.3f}")

Extracted skills:
  skill: python                    confidence: 0.998
  skill: machine learning          confidence: 0.995
  skill: sql                       confidence: 0.994
  skill: databases                 confidence: 0.930
  skill: kubernetes                confidence: 0.996


In [29]:
def extract_skills(text, extractor, threshold=0.5):
    """
    Takes raw text, runs NER skill extraction,
    returns a list of unique skill strings above confidence threshold.
    """
    text  = clean_text(text)
    raw   = extractor(text)

    skills = []
    for entity in raw:
        if entity['score'] >= threshold:
            skill = entity['word'].strip().lower()
            # Skip subword tokens and very short extractions
        if skill.startswith('##') or len(skill) <= 1:
            continue
        if skill not in skills:
            skills.append(skill)
    return skills

# Test it
skills = extract_skills(sample_text, skill_extractor)
print("Clean skill list:", skills)

Clean skill list: ['python', 'machine learning', 'sql', 'databases', 'kubernetes']


In [23]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [30]:
def match_skills(resume_skills, jd_skills, model, threshold=0.85, partial_threshold=0.6):
    """
    Compares JD skills against resume skills using embedding similarity.
    Returns matched, partial matches, and missing skills.
    """
    if not resume_skills or not jd_skills:
        return [], [], jd_skills

    # Embed all skills
    resume_embeddings = model.encode(resume_skills)
    jd_embeddings     = model.encode(jd_skills)

    # Cosine similarity matrix — shape (n_jd_skills, n_resume_skills)
    sim_matrix = cosine_similarity(jd_embeddings, resume_embeddings)

    matched  = []
    partial  = []
    missing  = []

    for i, jd_skill in enumerate(jd_skills):
        best_score = sim_matrix[i].max()
        best_match = resume_skills[sim_matrix[i].argmax()]

        if best_score >= threshold:
            matched.append({
                'jd_skill'     : jd_skill,
                'resume_match' : best_match,
                'score'        : round(float(best_score), 3)
            })
        elif best_score >= partial_threshold:
            partial.append({
                'jd_skill'     : jd_skill,
                'resume_match' : best_match,
                'score'        : round(float(best_score), 3)
            })
        else:
            missing.append(jd_skill)

    return matched, partial, missing

print("match_skills function defined")

match_skills function defined


In [31]:
resume_skills = ['python', 'machine learning', 'sql', 'data analysis']
jd_skills     = ['python', 'deep learning', 'postgresql', 'docker', 'kubernetes']

matched, partial, missing = match_skills(resume_skills, jd_skills, embedding_model)

print("=== Skill Match Results ===\n")
print("MATCHED:")
for m in matched:
    print(f"  JD: {m['jd_skill']:<20} Resume: {m['resume_match']:<20} Score: {m['score']}")

print("\nPARTIAL MATCHES:")
for p in partial:
    print(f"  JD: {p['jd_skill']:<20} Resume: {p['resume_match']:<20} Score: {p['score']}")

print("\nMISSING:")
for s in missing:
    print(f"  {s}")

=== Skill Match Results ===

MATCHED:
  JD: python               Resume: python               Score: 1.0

PARTIAL MATCHES:
  JD: deep learning        Resume: machine learning     Score: 0.689

MISSING:
  postgresql
  docker
  kubernetes


In [33]:
def generate_gap_report(resume_text, jd_text, skill_extractor, embedding_model):
    """
    End-to-end gap analysis:
    1. Extract skills from resume and JD
    2. Match skills using embeddings
    3. Return structured gap report
    """
    # Extract skills
    resume_skills = extract_skills(resume_text, skill_extractor)
    jd_skills     = extract_skills(jd_text, skill_extractor)

    # Match skills
    matched, partial, missing = match_skills(
        resume_skills, jd_skills, embedding_model
    )

    # Build report
    report = {
        'resume_skills'  : resume_skills,
        'jd_skills'      : jd_skills,
        'matched'        : matched,
        'partial'        : partial,
        'missing'        : missing,
        'match_rate'     : round(len(matched) / len(jd_skills) * 100, 1) if jd_skills else 0
    }

    return report


def print_gap_report(report):
    print("=" * 55)
    print("GAP ANALYSIS REPORT")
    print("=" * 55)

    print(f"\nResume skills found : {report['resume_skills']}")
    print(f"JD required skills  : {report['jd_skills']}")
    print(f"Match rate          : {report['match_rate']}%")

    print("\n✓ MATCHED SKILLS:")
    if report['matched']:
        for m in report['matched']:
            print(f"  {m['jd_skill']:<20} ← {m['resume_match']} ({m['score']})")
    else:
        print("  None")

    print("\n~ PARTIAL MATCHES:")
    if report['partial']:
        for p in report['partial']:
            print(f"  {p['jd_skill']:<20} ≈ {p['resume_match']} ({p['score']})")
    else:
        print("  None")

    print("\n✗ MISSING SKILLS:")
    if report['missing']:
        for s in report['missing']:
            print(f"  {s}")
    else:
        print("  None — strong match!")

print("Gap report functions defined")

Gap report functions defined


In [34]:
import json
from datasets import load_dataset

dataset  = load_dataset(
    "cnamuangtoun/resume-job-description-fit",
    cache_dir=r"e:\TalentLens\notebooks\hf_cache"
)
test_df  = dataset['test'].to_pandas()

# Pick a Good Fit example
sample = test_df[test_df['label'] == 'Good Fit'].iloc[0]

report = generate_gap_report(
    resume_text    = sample['resume_text'],
    jd_text        = sample['job_description_text'],
    skill_extractor= skill_extractor,
    embedding_model= embedding_model
)

print_gap_report(report)

GAP ANALYSIS REPORT

Resume skills found : ['web', 'xpath', 'writing', 'selenium', 'testng', 'test', 'automation', 'web applications', 'working experience', 'apache', 'jenkins', 'continuous', 'integration', 'testing', 'client requirements', 'sdlc', 'stlc', 'alm', 'ca', 'java', 'linux', 'selenium webdriver', 'selenium id', 'maven', 'finance', 'software', 'etl', 'data', 'data quality', 'big data', 'big data quality', 'interfaces', 'environment', 'windows', 'centos', 'unix']
JD required skills  : ['education', 'learning', 'fintech', 'saas', 'software as a service', 'software', 'application development', 'reliability', 'fault', 'platform development', 'microservices', 'architecture', 'environment', 'tests', 'software development', 'design', 'scalability', 'data recovery', 'business requirements', 'agile', 'pair', 'programming']
Match rate          : 13.6%

✓ MATCHED SKILLS:
  software             ← software (1.0)
  environment          ← environment (1.0)
  tests                ← test (0.9